# D — PixelCNN couleur
**Amjad AAYD et [information masquée] — [information masquée]**

Adaptation du TP AE / VAE / CVAE / PixelCNN à **CIFAR-10**, selon la demande orale
rapportée par l'étudiant. « CIFAR dix » est l'interprétation retenue du nom entendu.
Le PDF d'origine utilise FashionMNIST et MNIST ; cette version utilise CIFAR-10
pour les quatre blocs. Les images restent en couleur, sans conversion en gris.

Lire de haut en bas : **objectif → code commenté → sortie → explication**.
Les classes Python ne sont utilisées que pour définir les réseaux PyTorch.
Les boucles d'apprentissage restent visibles et explicites.

In [ ]:
# Importer uniquement les bibliothèques utilisées dans ce notebook.
from pathlib import Path
import json
import time
import random
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Fixer les tirages pour pouvoir refaire une expérience comparable.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(6)

# Utiliser le GPU si ce PC en possède un ; sinon utiliser le processeur.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Calcul sur :", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

# Tous les fichiers créés restent dans le dossier de ce TP.
racine = Path.cwd()
(racine / "figures").mkdir(exist_ok=True)
(racine / "models").mkdir(exist_ok=True)
debut = time.time()

## Charger les images couleur
Une image contient **3 × 32 × 32 = 3 072 nombres** : rouge, vert et bleu.
La division par 255 place les intensités dans [0,1]. Le test est chargé séparément
et ne sert jamais à ajuster les poids. Les classes ne supervisent que le CVAE.

In [ ]:
# Télécharger les vraies images CIFAR-10 avec torchvision.
# Au prochain lancement, les fichiers locaux sont réutilisés.
train = CIFAR10(root="data", train=True, download=True)
test = CIFAR10(root="data", train=False, download=True)

# CIFAR stocke [image, hauteur, largeur, couleur].
# PyTorch attend [image, couleur, hauteur, largeur].
x_train = torch.tensor(train.data).permute(0, 3, 1, 2).float() / 255.0
x_test = torch.tensor(test.data).permute(0, 3, 1, 2).float() / 255.0
y_train = torch.tensor(train.targets)
y_test = torch.tensor(test.targets)

# Former des lots de 128 images pour l'apprentissage.
donnees_train = TensorDataset(x_train, y_train)
chargeur = DataLoader(donnees_train, batch_size=128, shuffle=True)
classes = ["avion", "voiture", "oiseau", "chat", "cerf",
           "chien", "grenouille", "cheval", "bateau", "camion"]
print("Apprentissage :", x_train.shape)
print("Test :", x_test.shape)
print("Classes :", classes)
assert x_train.shape == (50000, 3, 32, 32)
assert x_test.shape == (10000, 3, 32, 32)

In [ ]:
# Cette fonction sert seulement à afficher et sauvegarder une grille de couleurs.
# permute remet les couleurs en dernier, comme attendu par matplotlib.
def afficher(images, titre, fichier, colonnes=8):
    images = images.detach().cpu().clamp(0, 1)
    lignes = (len(images) + colonnes - 1) // colonnes
    figure, axes = plt.subplots(lignes, colonnes,
                               figsize=(colonnes * 1.4, lignes * 1.5),
                               squeeze=False)
    for numero, axe in enumerate(axes.flat):
        axe.axis("off")
        if numero < len(images):
            axe.imshow(images[numero].permute(1, 2, 0))
    figure.suptitle(titre)
    figure.tight_layout()
    chemin = racine / "figures" / fichier
    figure.savefig(chemin, dpi=140, bbox_inches="tight")
    plt.close(figure)
    display(Image(filename=str(chemin)))

# Une seconde petite fonction affiche une courbe déjà calculée.
def courbe(valeurs, titre, fichier):
    plt.figure(figsize=(6, 3))
    plt.plot(range(1, len(valeurs) + 1), valeurs, marker="o")
    plt.xlabel("Époque")
    plt.ylabel("Perte moyenne")
    plt.title(titre)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    chemin = racine / "figures" / fichier
    plt.savefig(chemin, dpi=140)
    plt.close()
    display(Image(filename=str(chemin)))

## Pourquoi les couleurs demandent une adaptation
Un pixel contient trois valeurs : R, G, B. Pour conserver des masques simples et
une causalité correcte, on range chaque ligne ainsi : **R0,G0,B0,R1,G1,B1,…**.
Le réseau traite donc une grille **32 × 96** de valeurs, sans perdre aucune couleur.
L'ordre génératif devient R puis G puis B pour chaque pixel. Ce n'est pas le
PixelCNN RGB à masques de groupes de canaux ; c'est une adaptation pédagogique
par sérialisation des couleurs. Le voisinage horizontal est modifié et reste limité.
La dernière conversion restitue exactement les images RGB 32 × 32.

In [ ]:
# Sérialiser les trois canaux dans chaque ligne, sans convertir en niveaux de gris.
def vers_sequence(images):
    couleurs_en_dernier = images.permute(0, 2, 3, 1)
    return couleurs_en_dernier.reshape(len(images), 1, 32, 96)

# Effectuer l'opération inverse pour afficher les vraies couleurs.
def vers_rgb(sequence):
    couleurs_en_dernier = sequence.reshape(len(sequence), 32, 32, 3)
    return couleurs_en_dernier.permute(0, 3, 1, 2)

# Vérifier qu'aucune information n'est perdue par ce changement de rangement.
assert torch.equal(vers_rgb(vers_sequence(x_test[:8])), x_test[:8])
print("Conversion RGB aller-retour : exacte")

In [ ]:
# La seule couche personnalisée est celle que le TP demande : MaskedConv2d.
class MaskedConv2d(nn.Conv2d):
    def __init__(self, type_masque, entree, sortie, taille):
        super().__init__(entree, sortie, taille, padding=taille // 2)
        masque = torch.ones_like(self.weight)
        centre = taille // 2
        # Les lignes suivantes ne doivent jamais être visibles.
        masque[:, :, centre + 1:, :] = 0
        # A exclut aussi la valeur courante ; B l'autorise dans les couches cachées.
        if type_masque == "A":
            masque[:, :, centre, centre:] = 0
        else:
            masque[:, :, centre, centre + 1:] = 0
        self.register_buffer("masque", masque)

    def forward(self, x):
        poids_autorises = self.weight * self.masque
        return F.conv2d(x, poids_autorises, self.bias, padding=self.padding)

# La sortie a 256 scores pour chaque valeur de couleur à prédire.
class PixelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.reseau = nn.Sequential(
            MaskedConv2d("A", 1, 32, 7), nn.ReLU(),
            MaskedConv2d("B", 32, 32, 3), nn.ReLU(),
            MaskedConv2d("B", 32, 32, 3), nn.ReLU(),
            MaskedConv2d("B", 32, 32, 3), nn.ReLU(),
            nn.Conv2d(32, 256, kernel_size=1)
        )

    def forward(self, x):
        return self.reseau(x)

In [ ]:
# Montrer quels emplacements sont autorisés par les deux masques.
figure, axes = plt.subplots(1, 2, figsize=(6, 3))
for axe, nom in zip(axes, ["A", "B"]):
    couche = MaskedConv2d(nom, 1, 1, 7)
    axe.imshow(couche.masque[0, 0], cmap="Blues", vmin=0, vmax=1)
    axe.set_title("Masque " + nom)
figure.tight_layout()
chemin = racine / "figures" / "D_masques.png"
figure.savefig(chemin, dpi=140)
plt.close(figure)
display(Image(filename=str(chemin)))

## Entraîner avec 256 niveaux par composante de couleur
Les cibles sont des entiers 0–255. L'entrée reste normalisée dans [0,1].
Le sous-ensemble d'apprentissage de **12 000 images** est déclaré : le coût est
plus élevé qu'en gris, avec trois fois plus de valeurs par image.
Les autres modèles utilisent les 50 000 images ; aucun modèle n'apprend sur le test.

In [ ]:
# Sélectionner 12 000 images d'apprentissage de manière reproductible.
generateur = torch.Generator().manual_seed(42)
indices = torch.randperm(len(x_train), generator=generateur)[:12000]
sequence_train = vers_sequence(x_train[indices])
chargeur_pixel = DataLoader(TensorDataset(sequence_train), batch_size=64, shuffle=True)
modele = PixelCNN().to(device)
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.001)
pertes = []
for epoque in range(10):
    modele.train()
    somme = 0.0
    for lot in chargeur_pixel:
        images = lot[0].to(device)
        cibles = (images[:, 0] * 255).round().long()
        optimiseur.zero_grad()
        logits = modele(images)
        erreur = F.cross_entropy(logits, cibles)
        erreur.backward()
        optimiseur.step()
        somme += erreur.item() * len(images)
    moyenne = somme / len(sequence_train)
    pertes.append(moyenne)
    print("PixelCNN époque", epoque + 1, "nats/valeur RGB", moyenne, flush=True)
modele.eval()
torch.save(modele.state_dict(), racine / "models" / "PixelCNN.pt")
courbe(pertes, "PixelCNN : entropie croisée", "D_perte.png")

In [ ]:
# Vérifier la causalité du modèle entraîné, à une position de la séquence RGB.
# Modifier la valeur courante et les suivantes ne doit pas changer sa prédiction.
a = vers_sequence(x_test[:1]).to(device)
b = a.clone()
b[:, :, 14, 45:] = 1 - b[:, :, 14, 45:]
b[:, :, 15:, :] = 1 - b[:, :, 15:, :]
with torch.no_grad():
    avant = modele(a)[:, :, 14, 45]
    apres = modele(b)[:, :, 14, 45]
    difference = (avant - apres).abs().max().item()
assert difference < 0.000001
print("Différence du test causal :", difference)

## Prédiction avec contexte réel
Pour chaque position, le modèle reçoit les couleurs précédentes de la vraie image.
L'argmax est la valeur la plus probable. Cette comparaison n'est pas une génération libre.

In [ ]:
# Prédire les intensités, puis remettre les trois canaux à leur place.
with torch.no_grad():
    sequence = vers_sequence(x_test[:8]).to(device)
    valeurs = modele(sequence).argmax(dim=1).float() / 255.0
    predictions = vers_rgb(valeurs.unsqueeze(1)).cpu()
afficher(torch.cat([x_test[:8], predictions]),
         "Images réelles / prédictions avec contexte réel", "D_predictions.png")

# Mesurer la vraisemblance sur les 10 000 images test, sans apprentissage.
total = 0.0
with torch.no_grad():
    for images in x_test.split(64):
        sequence = vers_sequence(images).to(device)
        cibles = (sequence[:, 0] * 255).round().long()
        erreur = F.cross_entropy(modele(sequence), cibles)
        total += erreur.item() * len(images)
nats_par_valeur = total / len(x_test)
bits_par_valeur = nats_par_valeur / np.log(2)
print("Test :", bits_par_valeur, "bits par composante R, G ou B")

## Génération libre, une valeur après l'autre
On part d'un tableau de zéros. On tire R, puis G, puis B, et on continue à droite.
Les valeurs déjà tirées deviennent le contexte. Il faut **3 072 passes** pour une
grille, contre 784 pour les images grises du premier TP.

In [ ]:
# Créer huit nouvelles images sans fournir d'image réelle.
debut_generation = time.time()
sequence_generee = torch.zeros(8, 1, 32, 96, device=device)
with torch.no_grad():
    for ligne in range(32):
        for colonne in range(96):
            logits = modele(sequence_generee)[:, :, ligne, colonne]
            probabilites = torch.softmax(logits, dim=1)
            valeur = torch.multinomial(probabilites, num_samples=1).squeeze(1)
            sequence_generee[:, 0, ligne, colonne] = valeur.float() / 255.0
        if (ligne + 1) % 8 == 0:
            print("Génération : ligne", ligne + 1, "/ 32", flush=True)
images_generees = vers_rgb(sequence_generee).cpu()
secondes_generation = time.time() - debut_generation
afficher(images_generees, "PixelCNN : générations libres RGB", "D_generations.png")
print("Secondes pour huit images :", secondes_generation)

## Réponses aux trois questions D
1. Le masque A interdit de voir la valeur courante. Les couches B restent causales
parce qu'elles reçoivent des représentations déjà privées des valeurs futures.
2. La prochaine valeur dépend de celles déjà générées : on ne peut pas tirer toute
l'image en une seule passe avec cette méthode. L'entraînement reste parallèle.
3. PixelCNN optimise une vraisemblance discrète directement, mais génère lentement.
Un VAE génère en une passe et offre un code latent manipulable, mais peut lisser
les détails. La qualité des générations doit être regardée, pas déduite de la seule perte.

In [ ]:
# Enregistrer les mesures et les limites du protocole réellement exécuté.
resultats = {"pertes": pertes, "train_count": 12000, "epochs": 10,
             "test_nats_par_valeur": nats_par_valeur,
             "test_bits_par_valeur": bits_par_valeur,
             "causalite_difference": difference,
             "generation_8_images_secondes": secondes_generation,
             "secondes": time.time() - debut,
             "representation": "RGB serialise : 32 x 96, ordre R puis G puis B"}
(racine / "resultats_D.json").write_text(json.dumps(resultats, indent=2), encoding="utf-8")
print("Bloc D terminé.")